# Multilayer Coupling Schemes: spread vs. couple-to-self

When Infomap builds the state network for a multilayer network, it chooses **what an inter-layer relaxation link points to**:

- **Spread to neighbours (default).** Relaxation from `(layer1, n)` lands on a *neighbour* of `n` in the target layer. One inter-layer link per out-neighbour per layer pair: `O(L^2 * k)` links.
- **Couple physical nodes only (`--multilayer-relax-to-self`).** Relaxation lands on the *same physical node* `(layer2, n)`. One diagonal link per reachable layer: `O(L * k)` links.

The diagonal scheme builds a much smaller state network. With a flow correction (below) it also **reconstructs the spread model's flow**, so it returns the *same partition* as spread at a fraction of the links, memory, and runtime. This notebook shows that equivalence and the cost saving.

## Why it is equivalent to spread

Spread fuses the relaxation with an intra-step (jump to the target layer *and* step to a neighbour in one move). Couple-to-self splits that into a diagonal jump `(i,n) → (j,n)` plus an intra-step. Infomap computes the flow with a **two-step random walk** in which the diagonal node is *transient*: its flow is relayed through the target's intra links within the same step, so it accrues no visit. That fused step equals spread's transition exactly, so the stationary flow on the compact network is **bit-identical to spread's** (codelength matches to machine precision).

The diagonal state nodes stay coded, so each physical node's copies are held together by the **physical-node coding economy** of the state-network map equation (copies of `n` share `n`'s codeword within a module). This is the flow-model analog of Mucha-style diagonal inter-layer coupling, made bit-exactly equivalent to the default relax-to-neighbours model. (The same two-step construction Infomap already uses for bipartite-teleportation flow.)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import adjusted_mutual_info_score

from infomap import Infomap

%matplotlib inline


def make_shared(n_per_comm, K, L, within_deg, cross_deg, seed=0):
    """L layers sharing the same K planted communities (independent edge draws)."""
    rng = np.random.default_rng(seed)
    N = n_per_comm * K
    p_in = min(1.0, within_deg / max(1, n_per_comm - 1))
    p_out = cross_deg / max(1, (K - 1) * n_per_comm)
    comm = {nd: (nd - 1) // n_per_comm for nd in range(1, N + 1)}
    gt, intra = {}, []
    for layer in range(1, L + 1):
        for nd in range(1, N + 1):
            gt[(layer, nd)] = comm[nd]
        for i in range(1, N + 1):
            for j in range(i + 1, N + 1):
                if rng.random() < (p_in if comm[i] == comm[j] else p_out):
                    intra.append((layer, i, j, 1.0))
    return intra, gt


def run_scheme(intra, *, to_self, seed=123, num_trials=5):
    im = Infomap(silent=True, seed=seed, num_trials=num_trials,
                 multilayer_relax_to_self=to_self)
    for link in intra:
        im.add_multilayer_intra_link(*link)
    im.run()
    part = {(n.layer_id, n.node_id): n.module_id for n in im.nodes}
    return part, im.num_links, im.num_top_modules, im.codelength


def ami(a, b):
    keys = sorted(set(a) & set(b))
    return adjusted_mutual_info_score([a[k] for k in keys], [b[k] for k in keys])

## Same partition as spread

Across community counts, layer counts, sizes, and mixing, we compare the two schemes' partitions directly with the Adjusted Mutual Information (AMI). The flow is bit-exact, so the codelength matches to machine precision (`|Δ codelength|` ~ 1e-12) and the partition is identical, up to the optimizer occasionally finding an equal-quality alternative on the compact network.

In [ ]:
configs = [
    ('K3 L2 dense',  dict(n_per_comm=120, K=3, L=2, within_deg=12, cross_deg=2)),
    ('K4 L4 medium', dict(n_per_comm=120, K=4, L=4, within_deg=12, cross_deg=3)),
    ('K3 L6 sparse', dict(n_per_comm=200, K=3, L=6, within_deg=10, cross_deg=3)),
    ('K4 L4 mixed',  dict(n_per_comm=200, K=4, L=4, within_deg=10, cross_deg=4)),
]
rows = []
for name, cfg in configs:
    intra, _ = make_shared(seed=1, **cfg)
    sp_part, sp_links, sp_mods, sp_cl = run_scheme(intra, to_self=False)
    ts_part, ts_links, ts_mods, ts_cl = run_scheme(intra, to_self=True)
    rows.append({'network': name, 'AMI(spread,to-self)': round(ami(sp_part, ts_part), 3),
                 'modules spread': sp_mods, 'modules to-self': ts_mods,
                 '|d codelength|': abs(sp_cl - ts_cl),
                 'links spread': sp_links, 'links to-self': ts_links})
pd.DataFrame(rows).set_index('network')

## At a fraction of the cost

The diagonal scheme's inter-layer links scale `O(L * k)` against spread's `O(L^2 * k)`, so the state network is several times smaller and the gap grows with the number of layers `L`. On a 4000-node / layer network (avg degree 10) measured with the native CLI, at `L=10` this was about **3.3x faster and 3.2x less peak memory** (7.84s / 709 MB for spread vs 2.40s / 224 MB for to-self).

In [ ]:
layer_counts = [2, 4, 6, 8, 10]
links_spread, links_self = [], []
for L in layer_counts:
    intra, _ = make_shared(n_per_comm=40, K=3, L=L, within_deg=10, cross_deg=1, seed=1)
    links_spread.append(run_scheme(intra, to_self=False, num_trials=1)[1])
    links_self.append(run_scheme(intra, to_self=True, num_trials=1)[1])

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(layer_counts, links_spread, 'o-', label='spread (default)')
ax.plot(layer_counts, links_self, 's-', label='to-self')
ax.set_xlabel('number of layers L')
ax.set_ylabel('state-network links')
ax.set_title('State-network size vs number of layers')
ax.legend()
fig.tight_layout()

## Using it

- Enable with the `--multilayer-relax-to-self` CLI flag or `Infomap(multilayer_relax_to_self=True)`.
- Use it for large or many-layer multilayer networks where the default spread state network is expensive: you get the same partition with a much smaller network, lower peak memory, and faster runs, and the saving grows with the number of layers.
- The flow is bit-exact to spread (codelength matches to machine precision); the partition is identical up to the optimizer's stochastic search. The default remains spread for now; couple-to-self is the opt-in compact route to the same result.